In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
TRAIN_DIR = "/kaggle/input/competitions/sign-language-kaggle-comp/train"
TEST_DIR = "/kaggle/input/competitions/sign-language-kaggle-comp/test"
IMG_SIZE = 260 
BATCH_SIZE = 32
EPOCHS = 15

In [3]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [4]:
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
num_classes = len(train_dataset.classes)
class_names = train_dataset.classes

In [5]:
def get_convnext():
    model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    model.classifier[2] = nn.Linear(model.classifier[2].in_features, num_classes)
    return model.to(DEVICE)

def get_effnet():
    model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.DEFAULT)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model.to(DEVICE)

In [6]:
def train_model(model, name):
    print(f"\n--- Training {name} ---")
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=2e-4, 
                                            steps_per_epoch=len(train_loader), epochs=EPOCHS)
    
    for epoch in range(EPOCHS):
        model.train()
        correct, total = 0, 0
        loop = tqdm(train_loader, leave=False)
        for images, labels in loop:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
            loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
            loop.set_postfix(acc=correct/total)
    return model

In [7]:
model_conv = train_model(get_convnext(), "ConvNeXt")
model_eff = train_model(get_effnet(), "EfficientNetV2")

Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/torch/hub/checkpoints/convnext_tiny-983f1562.pth


100%|██████████| 109M/109M [00:00<00:00, 211MB/s]



--- Training ConvNeXt ---


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 197MB/s]



--- Training EfficientNetV2 ---


In [8]:
class TestDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = sorted(os.listdir(root_dir))
    def __len__(self): return len(self.image_files)
    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        image = Image.open(os.path.join(self.root_dir, img_name)).convert('RGB')
        if self.transform: image = self.transform(image)
        return image, img_name

test_loader = DataLoader(TestDataset(TEST_DIR, transform=test_transform), batch_size=BATCH_SIZE, shuffle=False)

In [9]:
model_conv.eval()
model_eff.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [10]:
torch.save(model_conv.state_dict(), 'model_convnext.pth')

torch.save(model_eff.state_dict(), 'model_effnet.pth')

In [11]:
results = []
print("\n--- Generating Submission with TTA ---")
with torch.no_grad():
    for images, filenames in tqdm(test_loader):
        images = images.to(DEVICE)
        
        out_conv = torch.softmax(model_conv(images), dim=1)
        out_eff = torch.softmax(model_eff(images), dim=1)
        
        images_flipped = torch.flip(images, dims=[3])
        out_conv_flip = torch.softmax(model_conv(images_flipped), dim=1)
        out_eff_flip = torch.softmax(model_eff(images_flipped), dim=1)
        
        final_probs = (out_conv * 0.3 + out_conv_flip * 0.3) + (out_eff * 0.2 + out_eff_flip * 0.2)
        preds = torch.argmax(final_probs, dim=1)
        
        for i in range(len(filenames)):
            results.append({
                "image_id": filenames[i],
                "label": class_names[preds[i].item()]
            })


--- Generating Submission with TTA ---


100%|██████████| 37/37 [00:36<00:00,  1.00it/s]


In [12]:
submission = pd.DataFrame(results)
submission.to_csv("submission.csv", index=False)
print("\nSubmission saved Head:\n", submission.head())


Submission saved Head:
          image_id label
0  img_00001.jpeg     N
1  img_00002.jpeg     J
2  img_00003.jpeg     Ö
3  img_00004.jpeg     T
4  img_00005.jpeg     J
